# Imported Required Spark libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType

# Created Spark Session

In [2]:
spark = SparkSession.builder \
    .appName("Spark Assignment Week 6") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


# Reading CSV file and using Spark DataFrame

In [3]:
df = spark.read.csv(
    "C:/Users/iamsu/OneDrive/Desktop/Assignment_6/data/ai_student_impact_dataset.csv",
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully")
df.show(5)

Dataset Loaded Successfully
+----------+--------------+-------------+----------------+------------------+--------------------+------------------------+--------------+-----------------+-----------------------+-----------------------+--------------------+--------------------------+-----------------+---------------------+------------------+
|Student_ID|Major_Category|Year_of_Study|Pre_Semester_GPA|Weekly_GenAI_Hours|    Primary_Use_Case|Prompt_Engineering_Skill|Tool_Diversity|Paid_Subscription|Traditional_Study_Hours|Perceived_AI_Dependency|Institutional_Policy|Anxiety_Level_During_Exams|Post_Semester_GPA|Skill_Retention_Score|Burnout_Risk_Level|
+----------+--------------+-------------+----------------+------------------+--------------------+------------------------+--------------+-----------------+-----------------------+-----------------------+--------------------+--------------------------+-----------------+---------------------+------------------+
|    100001|    Humanities|       Se

In [4]:
df.printSchema()

root
 |-- Student_ID: integer (nullable = true)
 |-- Major_Category: string (nullable = true)
 |-- Year_of_Study: string (nullable = true)
 |-- Pre_Semester_GPA: double (nullable = true)
 |-- Weekly_GenAI_Hours: double (nullable = true)
 |-- Primary_Use_Case: string (nullable = true)
 |-- Prompt_Engineering_Skill: string (nullable = true)
 |-- Tool_Diversity: integer (nullable = true)
 |-- Paid_Subscription: boolean (nullable = true)
 |-- Traditional_Study_Hours: double (nullable = true)
 |-- Perceived_AI_Dependency: integer (nullable = true)
 |-- Institutional_Policy: string (nullable = true)
 |-- Anxiety_Level_During_Exams: integer (nullable = true)
 |-- Post_Semester_GPA: double (nullable = true)
 |-- Skill_Retention_Score: double (nullable = true)
 |-- Burnout_Risk_Level: string (nullable = true)



# Selected the required columns

In [5]:
df = df.select(
    "Student_ID",
    "Major_Category",
    "Pre_Semester_GPA",
    "Weekly_GenAI_Hours",
    "Traditional_Study_Hours",
    "Post_Semester_GPA",
    "Skill_Retention_Score",
    "Anxiety_Level_During_Exams",
    "Burnout_Risk_Level"
)

# Filters students who study more than 15 hours per week

In [6]:
df.filter(df.Traditional_Study_Hours > 15).show()

+----------+--------------+----------------+------------------+-----------------------+-----------------+---------------------+--------------------------+------------------+
|Student_ID|Major_Category|Pre_Semester_GPA|Weekly_GenAI_Hours|Traditional_Study_Hours|Post_Semester_GPA|Skill_Retention_Score|Anxiety_Level_During_Exams|Burnout_Risk_Level|
+----------+--------------+----------------+------------------+-----------------------+-----------------+---------------------+--------------------------+------------------+
|    100002|       Medical|           3.821|              1.12|                  16.65|            3.696|                69.39|                         9|               Low|
|    100004|      Business|           3.789|              1.82|                  15.23|              4.0|                63.58|                         2|            Medium|
|    100008|          Arts|           2.746|              5.33|                  18.45|            2.965|                85.09|   

# Counting the total number of students in each major

In [7]:
df.groupBy("Major_Category").agg(count("*").alias("Total_Students")).show()

+--------------+--------------+
|Major_Category|Total_Students|
+--------------+--------------+
|    Humanities|          9994|
|          STEM|         15059|
|          Arts|          5933|
|      Business|         12538|
|       Medical|          6476|
+--------------+--------------+



# Calculated average pre-semester GPA for each major

In [8]:
df.groupBy("Major_Category").agg(avg("Pre_Semester_GPA").alias("Average_Pre_GPA")).show()

+--------------+------------------+
|Major_Category|   Average_Pre_GPA|
+--------------+------------------+
|    Humanities| 3.147849509705816|
|          STEM| 3.145851849392376|
|          Arts| 3.146786280128092|
|      Business|3.1417598500558297|
|       Medical| 3.151769302038296|
+--------------+------------------+



# Renamed Pre_Semester_GPA column

In [9]:
df = df.withColumnRenamed("Pre_Semester_GPA","Pre_GPA")

# Renamed Post_Semester_GPA column

In [10]:
df = df.withColumnRenamed("Post_Semester_GPA","Post_GPA")

In [11]:
print(df.columns)

['Student_ID', 'Major_Category', 'Pre_GPA', 'Weekly_GenAI_Hours', 'Traditional_Study_Hours', 'Post_GPA', 'Skill_Retention_Score', 'Anxiety_Level_During_Exams', 'Burnout_Risk_Level']


# Convertes GPA columns to Decimal type

In [12]:
df = df.withColumn("Pre_GPA",col("Pre_GPA").cast(DecimalType(3,2)))

df = df.withColumn("Post_GPA",col("Post_GPA").cast(DecimalType(3,2)))

# Convertes Weekly_GenAI_Hours to Integer

In [13]:
df = df.withColumn("Weekly_GenAI_Hours",col("Weekly_GenAI_Hours").cast("int"))

# Convertes Anxiety_Level_During_Exams to Double

In [14]:
df = df.withColumn("Anxiety_Level_During_Exams",col("Anxiety_Level_During_Exams").cast("double"))

In [15]:
df.printSchema()

root
 |-- Student_ID: integer (nullable = true)
 |-- Major_Category: string (nullable = true)
 |-- Pre_GPA: decimal(3,2) (nullable = true)
 |-- Weekly_GenAI_Hours: integer (nullable = true)
 |-- Traditional_Study_Hours: double (nullable = true)
 |-- Post_GPA: decimal(3,2) (nullable = true)
 |-- Skill_Retention_Score: double (nullable = true)
 |-- Anxiety_Level_During_Exams: double (nullable = true)
 |-- Burnout_Risk_Level: string (nullable = true)



# Creates a new column to calculate GPA improvement

In [16]:
df = df.withColumn("GPA_Improvement",round(col("Post_GPA") - col("Pre_GPA"),2))

# Calculates study hours to AI usage ratio

In [17]:
df = df.withColumn("Study_AI_Ratio",round(try_divide(col("Traditional_Study_Hours"),col("Weekly_GenAI_Hours")),2))

In [18]:
print(df.columns)

['Student_ID', 'Major_Category', 'Pre_GPA', 'Weekly_GenAI_Hours', 'Traditional_Study_Hours', 'Post_GPA', 'Skill_Retention_Score', 'Anxiety_Level_During_Exams', 'Burnout_Risk_Level', 'GPA_Improvement', 'Study_AI_Ratio']


In [19]:
df.count()

50000

In [20]:
df.show()

+----------+--------------+-------+------------------+-----------------------+--------+---------------------+--------------------------+------------------+---------------+--------------+
|Student_ID|Major_Category|Pre_GPA|Weekly_GenAI_Hours|Traditional_Study_Hours|Post_GPA|Skill_Retention_Score|Anxiety_Level_During_Exams|Burnout_Risk_Level|GPA_Improvement|Study_AI_Ratio|
+----------+--------------+-------+------------------+-----------------------+--------+---------------------+--------------------------+------------------+---------------+--------------+
|    100001|    Humanities|   2.42|                23|                   8.13|    2.39|                86.44|                       6.0|              High|          -0.03|          0.35|
|    100002|       Medical|   3.82|                 1|                  16.65|    3.70|                69.39|                       9.0|               Low|          -0.12|         16.65|
|    100003|      Business|   3.40|                21|           

# saving file in parquet type

In [23]:
df.write.mode("overwrite").parquet("C:\\Users\\iamsu\\OneDrive\\Desktop\\Assignment_6\\output\\output_parquet")

# Reading Parquet type 

In [49]:
parquet_df = spark.read.parquet("C:\\Users\\iamsu\\OneDrive\\Desktop\\Assignment_6\\output\\output_parquet")
parquet_df.show()

+----------+--------------+-------+------------------+-----------------------+--------+---------------------+--------------------------+------------------+---------------+--------------+
|Student_ID|Major_Category|Pre_GPA|Weekly_GenAI_Hours|Traditional_Study_Hours|Post_GPA|Skill_Retention_Score|Anxiety_Level_During_Exams|Burnout_Risk_Level|GPA_Improvement|Study_AI_Ratio|
+----------+--------------+-------+------------------+-----------------------+--------+---------------------+--------------------------+------------------+---------------+--------------+
|    100001|    Humanities|   2.42|                23|                   8.13|    2.39|                86.44|                       6.0|              High|          -0.03|          0.35|
|    100002|       Medical|   3.82|                 1|                  16.65|    3.70|                69.39|                       9.0|               Low|          -0.12|         16.65|
|    100003|      Business|   3.40|                21|           

# Count total rows in the Parquet dataset

In [50]:
print("Total Rows:", parquet_df.count())

Total Rows: 50000


# Checks null values in each column

In [ ]:
parquet_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in parquet_df.columns
]).describe().show()

# Removes rows containing null values

In [52]:
parquet_df = parquet_df.dropna()

# Counting rows after removing null values

In [54]:
print("Total Rows:", parquet_df.count())

Total Rows: 24444


# Filters students using AI for more than 5 hours per week

In [55]:
parquet_df = parquet_df.filter(col("Weekly_GenAI_Hours") > 5)

# Sorted students by GPA improvement in desc order

In [56]:
parquet_df = parquet_df.orderBy(col("GPA_Improvement").desc())


# Saved the final as a CSV file

In [57]:
parquet_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("C:\\Users\\iamsu\\OneDrive\\Desktop\\Assignment_6\\output\\output_csv\\Pipeline_Output_CSV")